# Text Transformations, Numbers, Punctuation, Stopwords, Stemming, Corpus

Where we process text in different ways in order to clean it up and make it more amenable to analysis.

In [1]:
from google.colab import drive
drive.mount('/content/drive')  # Add My Drive/<>

import os
os.chdir('drive/My Drive')
os.chdir('Books_Writings/NLPBook/')

Mounted at /content/drive


In [2]:
%%capture
!pip install ipypublish
!pip install cssselect

In [3]:
%pylab inline
import pandas as pd
import os
from ipypublish import nb_setup
%load_ext rpy2.ipython

Populating the interactive namespace from numpy and matplotlib


## Collect some text from the news

Here we use web scraping to collect news headlines from the Economic Times, an Indian newspaper.

It is useful to navigate to the URL for the newspaper to take a look at the page: https://economictimes.indiatimes.com

In [4]:
import requests
from lxml.html import fromstring

In [6]:
# Collect some text data
!pip install cssselect
import requests
from lxml.html import fromstring

#Copy the URL from the web site
url = 'https://economictimes.indiatimes.com'
html = requests.get(url, timeout=10).text

#See: http://infohost.nmt.edu/~shipman/soft/pylxml/web/etree-fromstring.html
doc = fromstring(html)

#http://lxml.de/cssselect.html#the-cssselect-method
x = doc.cssselect(".jsx-48c379259a10063f")
print(len(x))

headlines = [j.text_content() for j in x]
headlines = [j for j in headlines if len(j)>20]
headlines = unique(headlines)
headlines = headlines[:30]   #Needed to exclude any other stuff that was not needed.
for h in headlines:
    print(h)

56
A clear pattern emerges in UPI vs cards battle
Addressing Indiaâs cognitive time bomb
Air India partners with STARLUX Airlines
Ambani 'less well-off' this year: Forbes
Ambani 'less well-off' this year: ForbesSept third-hottest globally on recordWhat we know about the new Gaza dealHow Donald Trump pulled off his Gaza dealEarthquake of magnitude 3.1 strikes BhutanNo visas on the table with India: StarmerJSW MG wants to top India's luxe EV marketNational Employment Policy coming soonGovt plans pension law for coal workersDGCA slaps â¹20 lakh penalty on IndiGoPM hails Mumbai Metro Line-3 Phase 2BTCS Q2 earnings press conference called offA clear pattern emerges in UPI vs cards battleSkoda planning to launch EV in IndiaAddressing Indiaâs cognitive time bombAir India partners with STARLUX AirlinesArunachal Pradesh bans Coldrif cough syrupInternational students face fewer risks abroadBira plans $132 million fundraising: ReportIMC: India's telecom road map beyond 5GModi inaugurates Ind

## Remove punctuation from headlines

In [7]:
import string
print(string.punctuation)

punc_set = string.punctuation.replace('.','')
print(punc_set)

def removePuncStr(s):
    for c in string.punctuation:
        s = s.replace(c," ")
        s = s.replace('. ','')
    return s

def removePunc(text_array):
    return [removePuncStr(h) for h in text_array]

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
!"#$%&'()*+,-/:;<=>?@[\]^_`{|}~


In [8]:
headlines = removePunc(headlines)
headlines

['A clear pattern emerges in UPI vs cards battle',
 'Addressing Indiaâ\x80\x99s cognitive time bomb',
 'Air India partners with STARLUX Airlines',
 'Ambani  less well off  this year  Forbes',
 'Ambani  less well off  this year  ForbesSept third hottest globally on recordWhat we know about the new Gaza dealHow Donald Trump pulled off his Gaza dealEarthquake of magnitude 3 1 strikes BhutanNo visas on the table with India  StarmerJSW MG wants to top India s luxe EV marketNational Employment Policy coming soonGovt plans pension law for coal workersDGCA slaps â\x82¹20 lakh penalty on IndiGoPM hails Mumbai Metro Line 3 Phase 2BTCS Q2 earnings press conference called offA clear pattern emerges in UPI vs cards battleSkoda planning to launch EV in IndiaAddressing Indiaâ\x80\x99s cognitive time bombAir India partners with STARLUX AirlinesArunachal Pradesh bans Coldrif cough syrupInternational students face fewer risks abroadBira plans  132 million fundraising  ReportIMC  India s telecom road map

## Remove Numbers

In [9]:
def removeNumbersStr(s):
    for c in range(10):
        n = str(c)
        s = s.replace(n," ")
    return s

def removeNumbers(text_array):
    return [removeNumbersStr(h) for h in text_array]

In [10]:
headlines = removeNumbers(headlines)
headlines

['A clear pattern emerges in UPI vs cards battle',
 'Addressing Indiaâ\x80\x99s cognitive time bomb',
 'Air India partners with STARLUX Airlines',
 'Ambani  less well off  this year  Forbes',
 'Ambani  less well off  this year  ForbesSept third hottest globally on recordWhat we know about the new Gaza dealHow Donald Trump pulled off his Gaza dealEarthquake of magnitude     strikes BhutanNo visas on the table with India  StarmerJSW MG wants to top India s luxe EV marketNational Employment Policy coming soonGovt plans pension law for coal workersDGCA slaps â\x82¹   lakh penalty on IndiGoPM hails Mumbai Metro Line   Phase  BTCS Q  earnings press conference called offA clear pattern emerges in UPI vs cards battleSkoda planning to launch EV in IndiaAddressing Indiaâ\x80\x99s cognitive time bombAir India partners with STARLUX AirlinesArunachal Pradesh bans Coldrif cough syrupInternational students face fewer risks abroadBira plans      million fundraising  ReportIMC  India s telecom road map

## Remove Stopwords

Reference: https://pythonprogramming.net/stop-words-nltk-tutorial/

In [11]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def stopText(text_array):
    stop_words = set(stopwords.words('english'))
    stopped_text = []
    for h in text_array:
        words = word_tokenize(h)
        h2 = ''
        for w in words:
            if w.lower() not in stop_words:
                h2 = h2 + ' ' + w
        stopped_text.append(h2)
    return stopped_text

In [12]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
stopped_headlines = stopText(headlines)
stopped_headlines

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


[' clear pattern emerges UPI vs cards battle',
 ' Addressing Indiaâ\x80\x99s cognitive time bomb',
 ' Air India partners STARLUX Airlines',
 ' Ambani less well year Forbes',
 ' Ambani less well year ForbesSept third hottest globally recordWhat know new Gaza dealHow Donald Trump pulled Gaza dealEarthquake magnitude strikes BhutanNo visas table India StarmerJSW MG wants top India luxe EV marketNational Employment Policy coming soonGovt plans pension law coal workersDGCA slaps â\x82¹ lakh penalty IndiGoPM hails Mumbai Metro Line Phase BTCS Q earnings press conference called offA clear pattern emerges UPI vs cards battleSkoda planning launch EV IndiaAddressing Indiaâ\x80\x99s cognitive time bombAir India partners STARLUX AirlinesArunachal Pradesh bans Coldrif cough syrupInternational students face fewer risks abroadBira plans million fundraising ReportIMC India telecom road map beyond GModi inaugurates India Mobile Congress Indian seafood exporters troubled water',
 ' Ambani less well year

## Stemming

https://pythonprogramming.net/stemming-nltk-tutorial/

In [13]:
from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize

def stemText(text_array):
    stemmed_text = []
    for h in text_array:
        words = word_tokenize(h)
        h2 = ''
        for w in words:
            h2 = h2 + ' ' + PorterStemmer().stem(w)
        stemmed_text.append(h2)
    return stemmed_text

In [14]:
import nltk
nltk.download('punkt')
stemmed_headlines = stemText(headlines)
stemmed_headlines

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


[' a clear pattern emerg in upi vs card battl',
 ' address indiaâ\x80\x99 cognit time bomb',
 ' air india partner with starlux airlin',
 ' ambani less well off thi year forb',
 ' ambani less well off thi year forbessept third hottest global on recordwhat we know about the new gaza dealhow donald trump pull off hi gaza dealearthquak of magnitud strike bhutanno visa on the tabl with india starmerjsw mg want to top india s lux ev marketn employ polici come soongovt plan pension law for coal workersdgca slap â\x82¹ lakh penalti on indigopm hail mumbai metro line phase btc q earn press confer call offa clear pattern emerg in upi vs card battleskoda plan to launch ev in indiaaddress indiaâ\x80\x99 cognit time bombair india partner with starlux airlinesarunach pradesh ban coldrif cough syrupintern student face fewer risk abroadbira plan million fundrais reportimc india s telecom road map beyond gmodi inaugur india mobil congress indian seafood export in troubl water',
 ' ambani less well off 

## Write all docs to separate text files

This a typical approach in the text mining community. When creating a repository of plain text documents, each document is written as a separate text file to a folder. We do this here, so that we can see how to ingest a folder of such documents into a **corpus**, which is defined below.

In [15]:
def write2textfile(s,filename):
    text_file = open(filename, "w")
    text_file.write(s)
    text_file.close()

In [16]:
import os
os.system('mkdir CTEXT')

j = 0
for h in headlines:
    j = j + 1
    fname = "CTEXT/" + str(j) + ".ctxt"  #using "ctxt" to denote a corpus related file
    write2textfile(h,fname)

## Create a Corpus

A corpus is a data structure that contains multiple documents.

Functions may be written at the corpus level itself.

We only need to point to the folder and the function `PlaintextCorpusReader` in NLTK does the trick of constructng the corpus in one line of code.

In [17]:
#Read in the corpus
import nltk
from nltk.corpus import PlaintextCorpusReader
corpus_root = 'CTEXT/'
ctext = PlaintextCorpusReader(corpus_root, '.*')
ctext

<PlaintextCorpusReader in '/content/drive/MyDrive/Books_Writings/NLPBook/CTEXT'>

In [18]:
ctext.fileids()

['1.ctxt',
 '10.ctxt',
 '11.ctxt',
 '12.ctxt',
 '13.ctxt',
 '14.ctxt',
 '15.ctxt',
 '16.ctxt',
 '17.ctxt',
 '18.ctxt',
 '19.ctxt',
 '2.ctxt',
 '20.ctxt',
 '21.ctxt',
 '22.ctxt',
 '23.ctxt',
 '24.ctxt',
 '3.ctxt',
 '4.ctxt',
 '5.ctxt',
 '6.ctxt',
 '7.ctxt',
 '8.ctxt',
 '9.ctxt']

In [19]:
# We now have functions that apply to the entire corpus
print(ctext.words(), len(ctext.words()))
print(len(set(ctext.words()))) # gives the vocabulary
print(ctext.words('1.ctxt'), len(ctext.words('1.ctxt')))

['A', 'clear', 'pattern', 'emerges', 'in', 'UPI', 'vs', ...] 422
158
['A', 'clear', 'pattern', 'emerges', 'in', 'UPI', 'vs', ...] 9


In [20]:
ctext.words('2.ctxt')

['Addressing', 'Indiaâ', '\x80\x99', 's', 'cognitive', ...]